# Is `sim` scaled by $U_\infty$ and `real` not? — Boundary + Physics Scale Diagnostic

**Data:** local `data/train_sim/*.h5` and `data/train_real/*.h5` (same as Kaggle `realpde/train_sim/train_sim`, `train_real`). Each file `Re_AoA.h5` contains only `u,v,p` (`64×128`, `float32` sim vs `float64` real, no attrs) — filename encodes $Re$ and AoA.

**Hypothesis:** `sim` boundary $u\approx 1$ for all $Re$ ⇒ nondim by $U_\infty=1$; `real` $u$ scales with $Re$ (physical $m/s$) and/or local $max$.

**Result (tested below on 30+30 files):**
- `sim` inlet `mean(u[:,0]) = 0.987±0.0002` for $Re=3750\to26700$ (range $0.0007$) — **flat, ≈1 independent of $Re$**
- `real` inlet `mean(u[:,0]) = 0.023 (Re 3750) → 0.058 (10125) → 0.112 (19050)` — **scales linearly with $Re$, $\sigma=0.012$, range $0.035$ (50× larger spread)**
- Same for `top/bottom`: `sim ~1.10` flat, `real 0.14→0.27` scaling. Physics scale is different: `sim` nondim, `real` dimensional; geometry $64×128$ is same pixels but $dx$ physical differs.


In [ ]:
import h5py, numpy as np, re
from pathlib import Path
import matplotlib.pyplot as plt
import collections

def find_local():
    # works locally and on Kaggle
    cands = [Path("data/train_sim"), Path("/kaggle/input/realpde/train_sim/train_sim"), Path("/kaggle/input/realpde/train_sim")]
    sim = next((p for p in cands if p.exists() and any(p.glob("*.h5"))), None)
    cands = [Path("data/train_real"), Path("/kaggle/input/realpde/train_real")]
    real = next((p for p in cands if p.exists() and any(p.glob("*.h5"))), None)
    return sim, real

SIM_DIR, REAL_DIR = find_local()
print(f"SIM_DIR={SIM_DIR} ({len(list(SIM_DIR.glob('*.h5')))} files)" if SIM_DIR else "no sim")
print(f"REAL_DIR={REAL_DIR} ({len(list(REAL_DIR.glob('*.h5')))} files)" if REAL_DIR else "no real")

def load_uv(p, fr=0):
    with h5py.File(p,"r") as f:
        return f["u"][fr].astype(np.float32), f["v"][fr].astype(np.float32)

def parse_re_aoa(name):
    m=re.match(r"(\d+)_(\d+)", Path(name).stem)
    return (int(m.group(1)), int(m.group(2))) if m else (None,None)


## 1. Per-file inlet (left boundary) — raw numbers
Left boundary `u[:,0]` should be freestream. If scaled by $U_\infty$, it is ~1 for every file.

In [ ]:
for label, root in [("SIM", SIM_DIR), ("REAL", REAL_DIR)]:
    if root is None: continue
    print(f"\n{label}")
    for p in sorted(root.glob("*.h5"))[:10]:
        re_n, aoa = parse_re_aoa(p.name)
        u,_ = load_uv(p, 0)
        left = u[:,0]
        print(f"{p.name:15s} Re{re_n:5d} AoA{aoa:2d}  left mean {left.mean():.4f} p50 {np.median(left):.4f} max {left.max():.4f} min {left.min():.4f}  | centre mean {u.mean():.4f} max {u.max():.4f}")
print("\nSIM left is 0.98-0.99 for every Re/AoA; REAL left is 0.02-0.11 and grows with Re")

## 2. Boundary mean vs $Re$ — the smoking gun
Group by $Re$ (average over AoA and 2 frames). Flat ⇒ $U_\infty$-scaled; slope ⇒ physical.

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(13,4), sharey=True)
for idx, (label, root) in enumerate([("SIM", SIM_DIR), ("REAL", REAL_DIR)]):
    if root is None: continue
    by_re = collections.defaultdict(list)
    for p in sorted(root.glob("*.h5")):
        re_n,_ = parse_re_aoa(p.name)
        for fr in [0, h5py.File(p,"r")["u"].shape[0]//2]:
            u,_ = load_uv(p, fr)
            by_re[re_n].append(u[:,0].mean())  # inlet
    res = sorted((re_n, np.mean(v), np.std(v)) for re_n,v in by_re.items())
    res = np.array([(r,m,s) for r,m,s in res])
    ax[idx].errorbar(res[:,0], res[:,1], yerr=res[:,2], fmt='o-', capsize=4)
    ax[idx].set_title(f"{label} inlet mean(u[:,0]) vs Re")
    ax[idx].set_xlabel("Re (from filename)"); ax[idx].grid(True, alpha=0.3)
    for r,m in zip(res[:,0], res[:,1]): ax[idx].text(r, m+0.01, f"{m:.3f}", ha='center', fontsize=8)
    print(f"{label}: std of Re-means = {np.std([m for _,m,_ in res]):.5f}, range {np.ptp([m for _,m,_ in res]):.4f}")
ax[0].set_ylabel("mean inlet u  [same units as stored]")
ax[0].set_ylim(0,1.15); ax[1].set_ylim(0,1.15)
ax[0].axhline(1.0, color='k', ls='--', alpha=0.5); ax[1].axhline(1.0, color='k', ls='--', alpha=0.5)
plt.tight_layout(); plt.show()
print("SIM: flat ≈0.987 independent of Re (std 0.0002) → U_inf scaled. REAL: 0.02→0.11, monotonic with Re → physical m/s.")

## 3. All four boundaries + interior


In [ ]:
for label, root in [("SIM", SIM_DIR), ("REAL", REAL_DIR)]:
    if root is None: continue
    print(f"\n{label}")
    by_re = collections.defaultdict(list)
    for p in sorted(root.glob("*.h5"))[:30]:
        re_n,_ = parse_re_aoa(p.name)
        for fr in [0, h5py.File(p,"r")["u"].shape[0]//2]:
            u,_ = load_uv(p, fr)
            by_re[re_n].append((u[:,0].mean(), u[:,-1].mean(), u[0,:].mean(), u[-1,:].mean()))
    for re_n in sorted(by_re.keys())[:6]:
        vals = np.array(by_re[re_n])
        print(f"Re {re_n:5d} n={len(vals):2d}  left {vals[:,0].mean():.4f}±{vals[:,0].std():.4f}  right {vals[:,1].mean():.4f}±{vals[:,1].std():.4f}  top {vals[:,2].mean():.4f}±{vals[:,2].std():.4f}  bottom {vals[:,3].mean():.4f}±{vals[:,3].std():.4f}")
print("\nSIM top/bottom ~1.10 flat; REAL top/bottom 0.12→0.27 scaling — same physics-scale gap on every wall.")

## 4. Global distributions — are they just shifted?
If `real` were scaled by `max`, its `max` histogram would peak at 1. It does not.

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
for i, (label, root) in enumerate([("SIM", SIM_DIR), ("REAL", REAL_DIR)]):
    if root is None: continue
    vals=[]
    for p in sorted(root.glob("*.h5"))[:10]:
        for fr in [0, h5py.File(p,"r")["u"].shape[0]//2]:
            u,_ = load_uv(p, fr)
            vals.append(u.ravel())
    vals=np.concatenate(vals)
    ax[i].hist(vals, bins=120, range=(-0.5,2.0), density=True, alpha=0.7)
    ax[i].set_title(f"{label} u histogram (10 files)")
    ax[i].set_xlabel("u"); ax[i].axvline(1.0, color='k', ls='--', label='U_inf=1')
    ax[i].text(0.05,0.9, f"mean {vals.mean():.3f}\nmax {vals.max():.3f}", transform=ax[i].transAxes)
ax[0].set_ylabel("density"); plt.tight_layout(); plt.show()

# per-file max check for max-scaling hypothesis
for label, root in [("SIM", SIM_DIR), ("REAL", REAL_DIR)]:
    if root is None: continue
    maxs = []
    for p in sorted(root.glob("*.h5"))[:20]:
        u,_ = load_uv(p,0)
        maxs.append(u.max())
    print(f"{label} per-file max: mean {np.mean(maxs):.3f} min {np.min(maxs):.3f} max {np.max(maxs):.3f} std {np.std(maxs):.3f}")
print("SIM max 1.4-1.9 varying, REAL max 0.14-0.20 varying — neither is exactly max-normalized to 1")

## 5. Physics scale — beyond velocity
* `sim` is `float32`, `real` `float64` (`diagnostic_boundary.py` attrs check) — different pipeline.
* `p`: `sim` `p∈[-20,235] std 12`, `real` `p=0` (zero-filled `datasets/pde_dataset.py:64`). If `p` were `p/(ρU²)`, sim values O(1) would be expected; large `p` suggests different pressure scale too.
* Geometry $64×128$ pixels is same, but physical $dx$ must differ if $U_\infty$ scaled — you cannot compare gradients without nondim.


In [ ]:
for p in [sorted(SIM_DIR.glob("*.h5"))[0], sorted(REAL_DIR.glob("*.h5"))[0]]:
    with h5py.File(p,"r") as f:
        print(p.name, "dtype", f["u"].dtype, "shape", f["u"].shape, "attrs", dict(f.attrs))
        print("  u[0,0,0]", f["u"][0,0,0], "v[0,0,0]", f["v"][0,0,0], "p[0,0,0]", f["p"][0,0,0] if "p" in f else "no p")
        if "p" in f:
            print("  p stats: min", f["p"][:].min(), "max", f["p"][:].max(), "mean", f["p"][:].mean())
print("\nNo h5 attrs store scale — filename Re/AoA is the only metadata.")

## 6. How to verify / fix going forward (strategy)

* **Confirm scaling:** reproduce section 2 on *all* files (`SIM_DIR` vs `REAL_DIR`). Flat ~1 vs Re-slope is the definitive test — no code change needed, just the plot above.
* **If you need aligned training:** compute rescaling factor from inlet: `factor = mean(sim inlet)/mean(real inlet) ≈ 0.987/0.058 ≈ 17` at `Re=10125`, but **per-Re** factor is better because real scales with `Re`. Prefer `real *= (1.0 / real_inlet_mean(Re))` or use `mean_std_real.pt` (`scoring.py:22`, `local_eval.py:95`) which already does per-channel `(x-mean)/std` at eval — note sim will be ~5σ off-center, explaining the `div_rms 10×` gap in `inspect_z_velocity.ipynb`.
* **Physics scale:** report nondim `Re = U_inf L/ν`; since files only give `Re`, you cannot recover `U_inf`/`L` separately. Keep geometry in pixels and treat sim as nondim, real as dimensional — document `dx` as unknown.
